# Explainable IoT IDS — Full Pipeline

End-to-end reproducible run of all experiments for the paper. Runs on Google Colab free tier (CPU, no GPU required).

## What this notebook does

1. Installs pinned dependencies
2. Uploads the experiment scripts from `src/`
3. Acquires all three datasets (CICIDS2017, BoT-IoT, ToN-IoT)
4. Trains XGBoost, RF, DT baselines with hybrid MI+RF feature selection
5. Runs the full audit pipeline: faithfulness, stability, adversarial, SHAP-drift, counterfactuals, 5×2 CV, ablation, multiclass, efficiency
6. Collects all CSV tables and figures into a downloadable zip

**Expected runtime:** ~45–60 minutes on Colab free CPU.


## 1. Install dependencies

In [ ]:
!pip install -q numpy pandas pyarrow scikit-learn scipy xgboost shap lime matplotlib joblib tqdm adversarial-robustness-toolbox dice-ml
print('Dependencies installed. Restart runtime if prompted.')

## 2. Set up working directory

Upload these files to Colab (left panel → Files → Upload) before running:
- `cicids2017_clean.parquet`
- `ton_iot_binary.parquet`
- all `*.py` files from the repository's `src/` directory

Optionally upload pre-downloaded BoT-IoT CSVs to skip the download step.


In [ ]:
import os, shutil
from pathlib import Path

WORK = Path('/content/project')
(WORK / 'src').mkdir(parents=True, exist_ok=True)
(WORK / 'data' / 'raw' / 'bot_iot').mkdir(parents=True, exist_ok=True)
(WORK / 'outputs' / 'models').mkdir(parents=True, exist_ok=True)
(WORK / 'outputs' / 'tables').mkdir(parents=True, exist_ok=True)
(WORK / 'outputs' / 'figures').mkdir(parents=True, exist_ok=True)
(WORK / 'outputs' / 'logs').mkdir(parents=True, exist_ok=True)

# Move uploaded .py files into src/
for f in Path('/content').glob('*.py'):
    shutil.move(str(f), str(WORK / 'src' / f.name))
    print(f'moved src: {f.name}')

# Move uploaded parquet files into data/
for f in ['cicids2017_clean.parquet', 'ton_iot_binary.parquet', 'bot_iot_binary.parquet']:
    p = Path('/content') / f
    if p.exists():
        shutil.move(str(p), str(WORK / 'data' / f))
        print(f'moved data: {f}')

# Move any uploaded BoT-IoT raw CSVs
for f in Path('/content').glob('*BoT*.csv'):
    shutil.move(str(f), str(WORK / 'data' / 'raw' / 'bot_iot' / f.name))
    print(f'moved raw: {f.name}')
for f in Path('/content').glob('UNSW_*Iot*.csv'):
    shutil.move(str(f), str(WORK / 'data' / 'raw' / 'bot_iot' / f.name))
    print(f'moved raw: {f.name}')

os.chdir(WORK / 'src')
print('\nWorking dir:', os.getcwd())
print('Scripts present:', sorted([p.name for p in (WORK/'src').glob('*.py')]))
print('Data present:', sorted([p.name for p in (WORK/'data').iterdir()]))

## 3. Download BoT-IoT (if not already present)

If automated download fails, manually download from https://research.unsw.edu.au/projects/bot-iot-dataset, extract the zip, and upload the CSV files from "5%/10-best features/10-best Training-Testing split/" to Colab. Then re-run step 2.

In [ ]:
from pathlib import Path
bot_parquet = Path('/content/project/data/bot_iot_binary.parquet')
bot_raw = list(Path('/content/project/data/raw/bot_iot').glob('*.csv'))

if bot_parquet.exists():
    print(f'BoT-IoT parquet already exists ({bot_parquet.stat().st_size/1e6:.1f} MB). Skipping.')
elif bot_raw:
    print(f'Raw CSVs found: {len(bot_raw)} files. Running preprocessing...')
    !python preprocess_botiot.py
else:
    print('No BoT-IoT data found locally. Attempting download...')
    !python download_botiot.py
    if list(Path('/content/project/data/raw/bot_iot').glob('*.csv')):
        !python preprocess_botiot.py
    else:
        print('\nDownload failed. Please manually upload BoT-IoT CSVs to Colab (left panel) and re-run step 2.')

## 3b. Clear stale model caches

Force re-training of cached XGBoost bundles. Required whenever `data_utils.py`, `model_utils.py`, or any preprocessing code changes, otherwise the pipeline silently reuses the old (possibly leaky) bundle.

In [ ]:
import shutil
from pathlib import Path
models_dir = Path('/content/project/outputs/models')
if models_dir.exists():
    for f in models_dir.glob('*.joblib'):
        print(f'removing stale cache: {f.name}')
        f.unlink()
else:
    print('no cache directory yet; nothing to clear')

## 4. Train baseline models (XGB/RF/DT × 3 datasets)

In [ ]:
!python train_baselines.py

## 5. XAI faithfulness (SHAP + LIME)

~5-10 min. LIME is significantly slower than SHAP.

In [ ]:
!python faithfulness.py

## 6. XAI stability (30 bootstrap resamples)

In [ ]:
!python stability.py

## 7. Adversarial attacks (ZOO)

HopSkipJump is excluded due to known non-convergence against XGBoost tree ensembles (documented in the output CSV).

In [ ]:
!python adversarial.py

## 8. SHAP-drift adversarial detector

In [ ]:
!python shap_drift.py

## 9. Counterfactual explanations (DiCE)

In [ ]:
!python counterfactuals.py

## 10. 5x2 cross-validation + significance tests

In [ ]:
!python cv_significance.py

## 11. Multiclass classification (XGBoost)

CICIDS2017 and BoT-IoT only; ToN-IoT skipped (multiclass labels not available in current parquet).

In [ ]:
!python multiclass.py

## 12. Feature-selection ablation (MI-only / RF-only / Hybrid)

In [ ]:
!python ablation.py

## 13. Inference efficiency (model size + latency)

In [ ]:
!python efficiency.py

## 11. Inspect all output tables

In [ ]:
import pandas as pd
from pathlib import Path

tab = Path('/content/project/outputs/tables')
for csv in sorted(tab.glob('*.csv')):
    print(f'\n=== {csv.name} ===')
    try:
        df = pd.read_csv(csv)
        print(df.to_string(index=False))
    except Exception as e:
        print(f'(error reading: {e})')

## 12. Zip outputs and download

In [ ]:
import shutil
shutil.make_archive('/content/pipeline_outputs', 'zip', '/content/project/outputs')
print('Created /content/pipeline_outputs.zip')

from google.colab import files
files.download('/content/pipeline_outputs.zip')